This is prototype 1 with basic CNN:

In [25]:
# If you are running in a fresh environment, uncomment to install dependencies:
#!pip -q install torch torchvision matplotlib tqdm scikit-learn
# This is copied from HW2

import os
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import ssl
import certifi
from PIL import Image
from torch.utils.data import Dataset
from torch.utils.data import random_split
from torchvision import models

ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

print("torch:", torch.__version__)
device = "cpu"  # For fixed, reproducible results. (You may switch to "cuda" after you finish debugging.)
print("device:", device)

def set_seed(seed: int = 42):
    """Make results as reproducible as possible across runs."""
    import os, random
    import numpy as np
    import torch

    os.environ["PYTHONHASHSEED"] = str(seed)
    # If you later switch to CUDA and want maximal determinism:
    # os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Deterministic flags (safe on CPU; on GPU some ops may error if non-deterministic)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print("Warning: could not enable full deterministic algorithms:", e)

set_seed(42)

torch: 2.11.0
device: cpu


Loading the data

In [26]:
#  Old Dataloaders
''''''
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    root="ucsc-cse-144-spring-2026-final-project/train",
    transform=train_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [27]:
# Verifying Labels
print(train_dataset.class_to_idx)

{'0': 0, '1': 1, '10': 2, '11': 3, '12': 4, '13': 5, '14': 6, '15': 7, '16': 8, '17': 9, '18': 10, '19': 11, '2': 12, '20': 13, '21': 14, '22': 15, '23': 16, '24': 17, '25': 18, '26': 19, '27': 20, '28': 21, '29': 22, '3': 23, '30': 24, '31': 25, '32': 26, '33': 27, '34': 28, '35': 29, '36': 30, '37': 31, '38': 32, '39': 33, '4': 34, '40': 35, '41': 36, '42': 37, '43': 38, '44': 39, '45': 40, '46': 41, '47': 42, '48': 43, '49': 44, '5': 45, '50': 46, '51': 47, '52': 48, '53': 49, '54': 50, '55': 51, '56': 52, '57': 53, '58': 54, '59': 55, '6': 56, '60': 57, '61': 58, '62': 59, '63': 60, '64': 61, '65': 62, '66': 63, '67': 64, '68': 65, '69': 66, '7': 67, '70': 68, '71': 69, '72': 70, '73': 71, '74': 72, '75': 73, '76': 74, '77': 75, '78': 76, '79': 77, '8': 78, '80': 79, '81': 80, '82': 81, '83': 82, '84': 83, '85': 84, '86': 85, '87': 86, '88': 87, '89': 88, '9': 89, '90': 90, '91': 91, '92': 92, '93': 93, '94': 94, '95': 95, '96': 96, '97': 97, '98': 98, '99': 99}


In [28]:
# Create custom dataset 

class TestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform

        self.image_names = sorted(os.listdir(test_dir))

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]

        image_path = os.path.join(
            self.test_dir,
            image_name
        )

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, image_name

In [29]:
# Test dataset and loader

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

test_dataset = TestDataset(
    "ucsc-cse-144-spring-2026-final-project/test",
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [30]:
# Create training and validation split
from torch.utils.data import Subset


full_train_dataset = datasets.ImageFolder(
    root="ucsc-cse-144-spring-2026-final-project/train",
    transform=train_transform
)

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_subset, val_subset = random_split(
    full_train_dataset,
    [train_size, val_size]
)

train_dataset = datasets.ImageFolder(
    root="ucsc-cse-144-spring-2026-final-project/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root="ucsc-cse-144-spring-2026-final-project/train",
    transform=val_transform
)

train_dataset = Subset(
    train_dataset,
    train_subset.indices
)

val_dataset = Subset(
    val_dataset,
    val_subset.indices
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print(train_dataset[0][0].shape)
print(val_dataset[0][0].shape)

torch.Size([3, 224, 224])
torch.Size([3, 224, 224])


In [31]:
#Loading pretrained model

weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

num_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    num_features,
    100
)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
model = model.to(device)

#Loss + optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.1,
    patience=2
)

In [32]:
print(device)
print(torch.backends.mps.is_available())

mps
True


In [34]:
# Training + validation loop
best_val_acc = 0.0
best_epoch = 0

for epoch in range(30):

    ########################
    # TRAINING
    ########################

    model.train()

    running_train_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)

    ########################
    # VALIDATION
    ########################

    model.eval()

    running_val_loss = 0.0

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            # Predicted class
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    avg_val_loss = running_val_loss / len(val_loader)

    val_accuracy = 100 * correct / total
    
    if val_accuracy > best_val_acc:

        best_val_acc = val_accuracy
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

        print(f"Saved new best model! "
            f"Val Accuracy = {val_accuracy:.2f}%")

    ########################
    # PRINT RESULTS
    ########################

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f}")
    print(f"Val Accuracy: {val_accuracy:.2f}%")
    
print(f"Best validation accuracy: "
      f"{best_val_acc:.2f}% "
      f"(Epoch {best_epoch})")

Saved new best model! Val Accuracy = 59.26%
Epoch 1
Train Loss: 0.5846
Val Loss: 1.5065
Val Accuracy: 59.26%
Epoch 2
Train Loss: 0.5007
Val Loss: 1.4722
Val Accuracy: 59.26%
Epoch 3
Train Loss: 0.4568
Val Loss: 1.4369
Val Accuracy: 58.33%
Saved new best model! Val Accuracy = 61.57%
Epoch 4
Train Loss: 0.3961
Val Loss: 1.3820
Val Accuracy: 61.57%
Saved new best model! Val Accuracy = 62.04%
Epoch 5
Train Loss: 0.3351
Val Loss: 1.3771
Val Accuracy: 62.04%
Epoch 6
Train Loss: 0.2951
Val Loss: 1.3550
Val Accuracy: 59.72%
Epoch 7
Train Loss: 0.2563
Val Loss: 1.3334
Val Accuracy: 59.72%
Epoch 8
Train Loss: 0.2212
Val Loss: 1.3390
Val Accuracy: 60.19%
Epoch 9
Train Loss: 0.1875
Val Loss: 1.3289
Val Accuracy: 61.57%
Saved new best model! Val Accuracy = 62.50%
Epoch 10
Train Loss: 0.1818
Val Loss: 1.3245
Val Accuracy: 62.50%
Epoch 11
Train Loss: 0.1579
Val Loss: 1.3198
Val Accuracy: 59.72%
Epoch 12
Train Loss: 0.1502
Val Loss: 1.3021
Val Accuracy: 60.19%
Epoch 13
Train Loss: 0.1114
Val Loss: 1.3